# Your Zip Code is Your Health Sentence
## Food Deserts, Geographic Inequality, and Chronic Disease in America

**Team 2Kim** | SBU AI Community Datathon 2026 | Track: Healthcare & Wellness

---

## 1. Problem Statement and Hypothesis

Obesity and diabetes are commonly framed as lifestyle diseases — the result of individual choices about diet and exercise. But when your zip code determines whether a grocery store exists within walking distance, whether you can afford fresh food, and whether a doctor is nearby, *the choice was made before you were born.*

**Research Question:** Does living in a food desert independently predict higher rates of diabetes and obesity, even after controlling for income, insurance status, and race?

**Hypothesis:** Census tracts classified as food deserts by the USDA will show significantly higher diabetes and obesity prevalence than non-food-desert tracts, and this effect will persist after controlling for poverty rate and uninsured percentage — indicating that *food access itself* is a structural health determinant, not merely a proxy for poverty.

**Sub-questions:**
1. How much of the variance in diabetes rates is explained by geography (between-county) vs. individual tract factors (within-county)?
2. What is the life expectancy gap between the richest and poorest census tracts, and which variable dominates?
3. After controlling for income and food access, does race still independently predict diabetes prevalence?

## 2. Dataset Description

We merge five federal datasets at the census-tract level (joined on 11-digit FIPS codes):

| Dataset | Source | Geography | Key Variables |
|---------|--------|-----------|---------------|
| **USDA Food Access Research Atlas** (2019) | USDA ERS | Census tract (2010) | Food desert flags, low-access population shares, poverty rate |
| **CDC PLACES** (2025) | CDC | Census tract (2020) | Age-adjusted prevalence: obesity, diabetes, high BP, depression, CHD, physical inactivity, uninsured |
| **ACS 5-Year Estimates** (2022) | U.S. Census Bureau | Census tract (2020) | Median household income, poverty rate, race/ethnicity, education, population |
| **CDC Life Expectancy** (USALEEP, 2010-2015) | CDC NCHS | Census tract (2010) | Life expectancy at birth, standard error |
| **HRSA HPSA** (2026) | HRSA | County | Primary care shortage designation, HPSA score |

**Note on census tract vintages:** USDA and USALEEP use 2010 census tracts while PLACES and ACS use 2020 tracts. Most FIPS codes are stable across decades; we report match rates for transparency.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.loaders.merge import build_master
from src.analysis.health_stats import run_phase2, run_phase3, run_phase4
from src.analysis.health_index import build_health_disadvantage_index

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 6)})
%matplotlib inline

DATA_PROCESSED = Path("..") / "data" / "processed"

## 3. Data Cleaning and Preprocessing

In [ ]:
# Build or load the master dataset (downloads ~5 datasets on first run)
master_path = DATA_PROCESSED / "master.parquet"
if master_path.exists():
    df = pd.read_parquet(master_path)
    print(f"Loaded master: {df.shape[0]:,} rows x {df.shape[1]} cols")
else:
    df = build_master(save=True)
    print(f"Built master: {df.shape[0]:,} rows x {df.shape[1]} cols")

In [ ]:
# Cleaning steps applied in the pipeline:
# 1. Standardize all FIPS codes to 11-digit zero-padded strings
# 2. Filter out group-quarters tracts (military, prisons, dorms)
# 3. Use age-adjusted prevalence from CDC PLACES (controls for age distribution)
# 4. Compute derived columns: poverty_rate, pct_white/black/hispanic, pct_bachelors_plus
# 5. Create majority_race (40% threshold) and income_quintile

# Merge quality report
key_cols = [
    "is_food_desert", "obesity_pct", "diabetes_pct", "life_expectancy",
    "median_household_income", "poverty_rate", "pct_black", "uninsured_pct",
]
print("Key Column Coverage:")
print("-" * 40)
for col in key_cols:
    if col in df.columns:
        coverage = (1 - df[col].isna().mean()) * 100
        print(f"  {col:30s} {coverage:.1f}%")

# Usable analysis rows
usable = df.dropna(subset=["is_food_desert", "diabetes_pct", "obesity_pct"])
print(f"\nUsable tracts for core analysis: {len(usable):,} / {len(df):,}")

## 4. Exploratory Data Analysis

In [ ]:
# Health outcome distributions
health_cols = ["obesity_pct", "diabetes_pct", "high_bp_pct", "depression_pct",
               "chd_pct", "physical_inactivity_pct"]
available = [c for c in health_cols if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(available):
    ax = axes.flatten()[i]
    df[col].dropna().hist(bins=50, ax=ax, edgecolor="white", color="steelblue")
    ax.set_title(col.replace("_pct", " %").replace("_", " ").title(), fontsize=12)
    med = df[col].median()
    ax.axvline(med, color="red", linestyle="--", alpha=0.7, label=f"median={med:.1f}")
    ax.legend(fontsize=9)
for j in range(len(available), 6):
    axes.flatten()[j].set_visible(False)
plt.suptitle("Distribution of Health Outcomes Across Census Tracts", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Food desert vs non-food-desert comparison
compare_cols = ["diabetes_pct", "obesity_pct", "high_bp_pct", "life_expectancy"]
compare_cols = [c for c in compare_cols if c in df.columns]

fig, axes = plt.subplots(1, len(compare_cols), figsize=(5 * len(compare_cols), 5))
if len(compare_cols) == 1:
    axes = [axes]

for i, col in enumerate(compare_cols):
    data = df.dropna(subset=[col, "is_food_desert"])
    sns.boxplot(data=data, x="is_food_desert", y=col, ax=axes[i],
               palette=["#2ecc71", "#e74c3c"])
    axes[i].set_xticklabels(["Non-Food-Desert", "Food Desert"])
    axes[i].set_xlabel("")
    axes[i].set_title(col.replace("_pct", " %").replace("_", " ").title())
    for fd_val in [0, 1]:
        mean_val = data[data["is_food_desert"] == fd_val][col].mean()
        axes[i].text(fd_val, mean_val, f" \u03bc={mean_val:.1f}",
                    va="bottom", fontsize=9, fontweight="bold")

plt.suptitle("Health Outcomes: Food Desert vs Non-Food-Desert Tracts", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
corr_cols = [
    "diabetes_pct", "obesity_pct", "life_expectancy", "is_food_desert",
    "poverty_rate", "median_household_income", "pct_black", "pct_hispanic",
    "pct_bachelors_plus", "uninsured_pct", "physical_inactivity_pct",
]
corr_cols = [c for c in corr_cols if c in df.columns]

corr = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            ax=ax, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_title("Correlation Matrix: Health Outcomes x Socioeconomic Factors", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: poverty rate vs diabetes, colored by food desert status and by race
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sample = df.dropna(subset=["poverty_rate", "diabetes_pct", "is_food_desert"]).sample(
    min(5000, len(df)), random_state=42
)

for fd, color, label in [(0, "#2ecc71", "Non-Desert"), (1, "#e74c3c", "Food Desert")]:
    s = sample[sample["is_food_desert"] == fd]
    axes[0].scatter(s["poverty_rate"], s["diabetes_pct"], alpha=0.15, s=8, c=color, label=label)
axes[0].set_xlabel("Poverty Rate (%)")
axes[0].set_ylabel("Diabetes Prevalence (%)")
axes[0].set_title("Poverty vs Diabetes by Food Desert Status")
axes[0].legend()

if "majority_race" in df.columns:
    for race, color in [("White", "#3498db"), ("Black", "#e74c3c"), ("Hispanic", "#f39c12")]:
        s = sample[sample["majority_race"] == race]
        if len(s) > 0:
            axes[1].scatter(s["poverty_rate"], s["diabetes_pct"], alpha=0.15, s=8, c=color, label=race)
    axes[1].set_xlabel("Poverty Rate (%)")
    axes[1].set_ylabel("Diabetes Prevalence (%)")
    axes[1].set_title("Poverty vs Diabetes by Majority Race")
    axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Life expectancy by income quintile and race
if "life_expectancy" in df.columns and "income_quintile" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    le_data = df.dropna(subset=["life_expectancy", "income_quintile"])
    sns.boxplot(data=le_data, x="income_quintile", y="life_expectancy",
               ax=axes[0], palette="RdYlGn")
    axes[0].set_xlabel("Income Quintile (1=poorest, 5=richest)")
    axes[0].set_ylabel("Life Expectancy (years)")
    axes[0].set_title("Life Expectancy by Income Quintile")
    q_means = le_data.groupby("income_quintile")["life_expectancy"].mean()
    if len(q_means) >= 2:
        gap = q_means.iloc[-1] - q_means.iloc[0]
        axes[0].text(0.5, 0.02, f"Gap (Q5-Q1): {gap:.1f} years",
                    transform=axes[0].transAxes, fontsize=11, ha="center",
                    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))

    if "majority_race" in df.columns:
        le_race = df.dropna(subset=["life_expectancy", "income_quintile", "majority_race"])
        race_q = le_race.groupby(["income_quintile", "majority_race"])["life_expectancy"].mean().unstack()
        cols_to_plot = [c for c in ["Black", "Hispanic", "White"] if c in race_q.columns]
        race_q[cols_to_plot].plot(kind="bar", ax=axes[1], width=0.8)
        axes[1].set_xlabel("Income Quintile")
        axes[1].set_ylabel("Mean Life Expectancy")
        axes[1].set_title("Life Expectancy by Income x Race")
        axes[1].legend(title="Majority Race")
        axes[1].tick_params(axis="x", rotation=0)

    plt.tight_layout()
    plt.show()

In [ ]:
# Income x race -> diabetes heatmap
if all(c in df.columns for c in ["income_quintile", "majority_race", "diabetes_pct"]):
    matrix = df.dropna(subset=["income_quintile", "majority_race", "diabetes_pct"]).pivot_table(
        index="income_quintile", columns="majority_race",
        values="diabetes_pct", aggfunc="mean",
    )
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(matrix, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax,
                linewidths=1, cbar_kws={"label": "Mean Diabetes %"})
    ax.set_xlabel("Majority Race of Census Tract")
    ax.set_ylabel("Income Quintile (1=poorest, 5=richest)")
    ax.set_title("Diabetes Prevalence: Income Quintile x Majority Race")
    plt.tight_layout()
    plt.show()

## 5. Statistical Tests and Modeling

In [ ]:
# Phase 2: Food Access -> Chronic Disease
p2 = run_phase2(df)

if "odds_ratio_diabetes" in p2:
    od = p2["odds_ratio_diabetes"]
    print(f"\n{'='*60}")
    print(f"FINDING 1: Food desert residents have {od['odds_ratio']:.2f}x the odds")
    print(f"of above-median diabetes (95% CI: {od['ci_95_low']:.2f}-{od['ci_95_high']:.2f})")
    print(f"  Food desert mean diabetes:     {od['desert_mean_diabetes']}%")
    print(f"  Non-food-desert mean diabetes: {od['non_desert_mean_diabetes']}%")
    print(f"{'='*60}")

if "ols_diabetes" in p2:
    print(f"\nOLS Regression (DV: diabetes_pct), R2 = {p2['ols_diabetes']['r_squared']}")
    for var, v in p2["ols_diabetes"]["coefficients"].items():
        sig = "***" if v["p_value"] < 0.001 else "**" if v["p_value"] < 0.01 else "*" if v["p_value"] < 0.05 else ""
        print(f"  {var:25s} coef={v['coef']:+.4f}  p={v['p_value']} {sig}")

if "partial_corr_food_diabetes" in p2:
    pc = p2["partial_corr_food_diabetes"]
    print(f"\nPartial correlation (food_desert -> diabetes | income): r={pc['r']:.3f}, p={pc['p']}")

In [ ]:
# Phase 3: The Zip Code Effect
p3 = run_phase3(df)

if "variance_decomposition" in p3:
    vd = p3["variance_decomposition"]
    print(f"\nVariance Decomposition (diabetes_pct):")
    print(f"  Between-county: {vd['pct_between']:.1f}%")
    print(f"  Within-county:  {vd['pct_within']:.1f}%")

if "standardized_betas" in p3:
    print(f"\n{'='*60}")
    print("FINDING 2: Standardized Betas for Life Expectancy")
    for var, beta in sorted(p3["standardized_betas"].items(), key=lambda x: abs(x[1]), reverse=True):
        bar = '|' * int(abs(beta) * 40)
        print(f"  {var:30s} B={beta:+.3f}  {bar}")
    print(f"{'='*60}")

if "life_expectancy_gap" in p3:
    g = p3["life_expectancy_gap"]
    print(f"\nFINDING 3: Life expectancy gap (Q5-Q1): {g['gap_years']:.1f} years")
    print(f"  Poorest quintile: {g['poorest_quintile_mean_le']} years")
    print(f"  Richest quintile: {g['richest_quintile_mean_le']} years")

In [ ]:
# Phase 4: Race as a Residual Gap
p4 = run_phase4(df)

if "residual_analysis" in p4:
    ra = p4["residual_analysis"]
    print(f"\n{'='*60}")
    print(f"FINDING 4: Race remains {'SIGNIFICANT' if ra['race_still_significant'] else 'not significant'}")
    print(f"after controlling for income + food access")
    print(f"  R2 without race: {ra['r2_without_race']}")
    print(f"  R2 with race:    {ra['r2_with_race']}")
    print(f"  Delta-R2:        {ra['r2_change']}")
    print(f"  pct_black p-value: {ra['pct_black_p_value']}")
    print(f"{'='*60}")

if "cross_comparison" in p4:
    cc = p4["cross_comparison"]
    print(f"\nCross-comparison:")
    print(f"  High-income Black tracts: {cc['high_income_black_mean_diabetes']}% diabetes")
    print(f"  Low-income White tracts:  {cc['low_income_white_mean_diabetes']}% diabetes")
    closes = cc.get('income_closes_gap', False)
    print(f"  Income {'closes' if closes else 'DOES NOT close'} the racial gap")

In [ ]:
# Phase 5B: Health Disadvantage Index
p5 = build_health_disadvantage_index(df, phase2_results=p2, phase3_results=p3)

if "decile_gaps" in p5:
    print(f"\n{'='*60}")
    print(f"HEALTH DISADVANTAGE INDEX - Gaps (top vs bottom 10% tracts)")
    for metric, vals in p5["decile_gaps"].items():
        print(f"  {metric}: {vals['most_disadvantaged']} vs {vals['least_disadvantaged']} (gap: {vals['gap']})")
    print(f"{'='*60}")

if "path_diagram" in p5 and p5["path_diagram"]:
    print("\nPath Diagram Coefficients:")
    for path, vals in p5["path_diagram"].items():
        print(f"  {path}: B={vals['coef']}, R2={vals['r_squared']}")

## 6. Results and Conclusions

### Key Findings

1. **Food desert tracts have higher diabetes and obesity prevalence.** After controlling for poverty rate and uninsured percentage via OLS with heteroscedasticity-robust standard errors (HC1), food desert status remains a significant predictor. The ecological odds ratio for above-median tract-level diabetes in food deserts exceeds 1.0. Note: this is a *tract-level* association, not an individual-level risk estimate.

2. **Geography explains a substantial share of diabetes variance.** Between-county variance accounts for a meaningful fraction of total diabetes rate variance across tracts, confirming that where a tract is located — not just its individual characteristics — shapes health outcomes.

3. **Life expectancy gaps between income quintiles span years.** The richest census tracts have substantially higher mean life expectancy than the poorest. Standardized regression coefficients (years of life expectancy per 1-SD change in predictor) identify income and education as the dominant predictors.

4. **Race adds explanatory power beyond income and food access.** Adding `pct_black` to a model already containing poverty rate, food desert status, and uninsured percentage produces a statistically significant improvement in R². The cross-comparison of high-income majority-Black tracts vs. low-income majority-White tracts is exploratory (small sample sizes in some cells) and should be interpreted cautiously.

5. **The Health Disadvantage Index (descriptive, equal-weighted) reveals compounding inequality.** Tracts in the worst decile on the composite index show dramatically worse outcomes across every health metric. The index uses equal weights across food access, income, and healthcare access components to avoid circularity.

### Methodological Notes

- All OLS models use HC1 robust standard errors to address heteroscedasticity
- VIF diagnostics are computed for Phase 2 predictors to check for multicollinearity
- The odds ratio uses a median-split 2x2 table, which dichotomizes a continuous outcome — the Welch t-test of mean differences is the more conservative measure
- The path diagram shows bivariate associations along a hypothesized pathway, not a formal mediation analysis
- With ~15 hypothesis tests across all phases and no multiple-comparison correction, individual p-values should be interpreted with caution
- All analyses are unweighted by tract population — results reflect patterns across tracts, not across the U.S. population

## 7. Limitations and Future Work

### Limitations

1. **Ecological fallacy.** All data is at the census-tract level. We cannot infer that *individuals* in food deserts have higher diabetes — only that *tracts* do. The odds ratio is an ecological OR, not an individual-level risk estimate. Individual-level data (e.g., NHANES) would be needed to confirm.

2. **Census tract vintage mismatch.** USDA (2010 tracts) and CDC PLACES (2020 tracts) use different census geographies. While most FIPS codes are stable, approximately 5-15% of tracts may not match, reducing sample size. No crosswalk table is used. High-growth metros (Texas, Florida, Arizona) may lose 15-20% of tracts due to redistricting.

3. **Simplified race classification.** We classify tracts by plurality race among White, Black, and Hispanic only, with a 40% threshold. This erases Asian, Native American, multiracial, and other communities. Hispanic is an ethnicity (not mutually exclusive with race categories), which may inflate apparent percentages in some tracts.

4. **Correlation, not causation.** This is an observational, cross-sectional study. We cannot conclude that food deserts *cause* diabetes. Unmeasured confounders (healthcare quality, cultural diet patterns, environmental pollution) may explain part of the relationship.

5. **Spatial autocorrelation.** Neighboring census tracts are not independent — food deserts cluster geographically. This violates the independence assumption of OLS and inflates the effective sample size, making p-values anti-conservative. A spatial regression model (spatial lag or spatial error) would be more appropriate.

6. **Temporal misalignment.** Our five datasets span 2010-2025. The USDA food atlas (2019) predates the pandemic, while CDC PLACES (2025) reflects post-pandemic health. COVID-19 may have changed food access patterns and health outcomes in ways not captured.

7. **Multiple comparisons.** Approximately 15 hypothesis tests are run across all phases without Bonferroni or FDR correction. The family-wise error rate at alpha=0.05 with 15 tests is ~54%. Individual p-values should be interpreted cautiously.

8. **No population weighting.** All tracts are treated equally regardless of population. A rural tract with 1,500 residents has equal weight to an urban tract with 7,500. Results describe patterns across tracts, not across the U.S. population.

9. **Self-reported and modeled data.** CDC PLACES uses small-area estimation models, not direct measurement. HRSA shortage designations are administrative, not clinical.

10. **Path diagram is exploratory.** The four bivariate regressions in the path diagram show associations along a hypothesized pathway but do not constitute a formal mediation analysis. Coefficients cannot be multiplied to estimate indirect effects.

### Future Work

- Link to individual-level NHANES data to validate tract-level patterns
- Implement spatial regression models (spatial lag/error) to account for geographic clustering
- Use census tract crosswalk tables for precise 2010-to-2020 tract mapping
- Apply population weights to regressions for population-level inference
- Conduct formal mediation analysis (bootstrap-based or Sobel test) for the poverty-to-diabetes pathway
- Apply Benjamini-Hochberg FDR correction across all hypothesis tests
- Examine intervention data: do tracts that gained a grocery store show improved health outcomes?

## Dataset Citations (MLA 8)

United States Department of Agriculture, Economic Research Service. "Food Access Research Atlas." *USDA ERS*, 27 Apr. 2021, www.ers.usda.gov/data-products/food-access-research-atlas/. Accessed 28 Mar. 2026.

Centers for Disease Control and Prevention. "PLACES: Local Data for Better Health, Census Tract Data, 2025 Release." *CDC Data Portal*, data.cdc.gov/500-Cities-Places/PLACES-Local-Data-for-Better-Health-Census-Tract-D/cwsq-ngmh. Accessed 28 Mar. 2026.

United States Census Bureau. "American Community Survey 5-Year Estimates, 2018-2022." *Census Bureau API*, api.census.gov/data/2022/acs/acs5. Accessed 28 Mar. 2026.

Centers for Disease Control and Prevention, National Center for Health Statistics. "U.S. Life Expectancy at Birth by State and Census Tract, 2010-2015." *CDC Data Portal*, data.cdc.gov/NCHS/U-S-Life-Expectancy-at-Birth-by-State-and-Census-T/5h56-n989. Accessed 28 Mar. 2026.

Health Resources and Services Administration. "Health Professional Shortage Areas: Primary Care." *HRSA Data Warehouse*, data.hrsa.gov/DataDownload/DD_Files/BCD_HPSA_FCT_DET_PC.csv. Accessed 28 Mar. 2026.